# Project01 Tích hợp hệ đa tác tử và demo

Mục tiêu: chạy toàn bộ hội chẩn, quan sát thông điệp, thử dữ liệu thiếu, nhánh cảnh báo và veto của Council. Cần chạy notebook 01 và 03 trước nếu không có artifact đi kèm.

In [1]:
# Chạy được khi kernel bắt đầu ở PROJECT_01 hoặc notebooks.
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "diabetes_agent").exists() else Path.cwd().parent
assert (ROOT / "diabetes_agent").exists(), "Mở notebook từ thư mục PROJECT_01"
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from diabetes_agent.config import ARTIFACTS, DATA, FEATURES
print("Python:", sys.version.split()[0], "| Project:", ROOT.name)

Python: 3.11.9 | Project: PROJECT_01


## 1 Khởi tạo hệ thống và hồ sơ minh họa
Coordinator giao việc qua MessageBus trong cùng tiến trình. Sáu chuyên gia là Intake, Risk, Knowledge, Reminder, Council và Advisor. Đây là bảy vai trò tính cả điều phối, không phải bảy LLM độc lập.

In [2]:
from diabetes_agent.system import DiabetesSystem, review
from diabetes_agent.evaluation import example_profile, evaluate_all
system = DiabetesSystem()
profile = example_profile()
print("Hồ sơ minh họa thuộc train, không phải kiểm định:")
display(pd.Series(profile))

Hồ sơ minh họa thuộc train, không phải kiểm định:


HighBP                   0.0
HighChol                 1.0
CholCheck                1.0
BMI                     21.0
Smoker                   1.0
Stroke                   0.0
HeartDiseaseorAttack     0.0
PhysActivity             0.0
Fruits                   0.0
Veggies                  0.0
HvyAlcoholConsump        0.0
AnyHealthcare            1.0
NoDocbcCost              0.0
GenHlth                  3.0
MentHlth                 0.0
PhysHlth                 0.0
DiffWalk                 0.0
Sex                      0.0
Age                     13.0
Education                4.0
Income                   3.0
dtype: float64

## 2 Một lượt đầy đủ
Intake kiểm tra phạm vi trước. Risk phân loại vector khảo sát; Knowledge tìm bằng chứng; Reminder chọn hành động; Council kiểm tra hợp đồng; Advisor ghép kết quả có nguồn.

In [3]:
result = system.run("HbA1c phản ánh điều gì và ghi nhật ký ra sao?", profile,
                    context={"gap_days": 2, "fatigue": 0})
print(result["text"])
print("Trạng thái:", result["status"], "| Council:", result["review"])
display(pd.DataFrame(result["trace"]))
assert result["status"] == "completed" and len(result["trace"]) == 12

Mô hình random_forest: điểm 0.147, ngưỡng 0.230, dự đoán nhãn 0. Điểm mô hình cho nhãn gộp tiền tiểu đường/tiểu đường trong khảo sát UCI; không phải xác suất mắc bệnh trong tương lai.

HbA1c phản ánh đường huyết trung bình khoảng ba tháng, khác số đo tức thời. Cần đọc kết quả cùng bối cảnh và hướng dẫn chuyên môn. [D06]

Theo dõi và trao đổi kết quả với nhóm chăm sóc giúp quản lý bệnh. Project gợi ý ghi giá trị, thời gian, đơn vị và bối cảnh; lịch đo cần thống nhất với nhân viên y tế. [D08]

Type 1, type 2 và tiểu đường thai kỳ có cơ chế và hoàn cảnh khác nhau. Không suy ra loại bệnh từ điểm dự đoán nhị phân của project. [D02]

Nhắc việc mô phỏng: Gửi gợi ý ghi chép ngắn

Bạn có thể mang thông tin và câu hỏi tới nhân viên y tế để được đánh giá phù hợp. Sản phẩm học tập; không chẩn đoán, kê đơn hoặc thay đổi điều trị.
Trạng thái: completed | Council: {'status': 'approved', 'reasons': []}


,sender,receiver,performative,task,status,elapsed_ms
0,Coordinator,Intake,REQUEST,CHECK_SCOPE,NaN,NaN
1,Intake,Coordinator,INFORM,CHECK_SCOPE,completed,0.053
2,Coordinator,Risk,REQUEST,PREDICT_LABEL,NaN,NaN
3,Risk,Coordinator,INFORM,PREDICT_LABEL,completed,87.466
4,Coordinator,Knowledge,REQUEST,RETRIEVE,NaN,NaN
5,Knowledge,Coordinator,INFORM,RETRIEVE,completed,6.617
6,Coordinator,Reminder,REQUEST,PROPOSE_REMINDER,NaN,NaN
7,Reminder,Coordinator,INFORM,PROPOSE_REMINDER,completed,0.035
8,Coordinator,Council,REQUEST,VERIFY,NaN,NaN
9,Council,Coordinator,INFORM,VERIFY,approved,0.018


## 3 Dữ liệu thiếu không được tự điền
Model cần đúng 21 trường. Yêu cầu bổ sung không có nghĩa đã đưa ra dự đoán từ hồ sơ hoàn chỉnh.

In [4]:
incomplete = system.run("Phân loại", {"BMI": 26}, task="risk")
print(incomplete["status"])
print(incomplete["text"])
assert incomplete["status"] == "needs_input"

needs_input
Chưa chạy dự đoán: thiếu HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Education, Income

Bạn có thể mang thông tin và câu hỏi tới nhân viên y tế để được đánh giá phù hợp. Sản phẩm học tập; không chẩn đoán, kê đơn hoặc thay đổi điều trị.


## 4 Nhánh khẩn cấp ưu tiên trước mô hình
Số đo glucose không thuộc input CDC. Luật dưới 70 mg/dL có nguồn CDC và chỉ kích hoạt thông điệp cần xử trí/hỗ trợ, không chẩn đoán từ mô hình.

In [5]:
urgent = system.run("Tôi hỏi về HbA1c", profile, context={"glucose_mg_dl": 62})
print(urgent["status"], urgent["text"])
display(pd.DataFrame(urgent["trace"]))
assert len(urgent["trace"]) == 2
assert not any(m["receiver"] == "Risk" for m in urgent["trace"])

urgent Số đo dưới 70 mg/dL cần được xử trí kịp thời theo hướng dẫn của nhóm chăm sóc. Liên hệ hỗ trợ y tế; nếu không tỉnh táo hoặc không thể tự xử trí, cần trợ giúp khẩn cấp.


,sender,receiver,performative,task,status,elapsed_ms
0,Coordinator,Intake,REQUEST,CHECK_SCOPE,NaN,NaN
1,Intake,Coordinator,INFORM,CHECK_SCOPE,urgent,0.036


## 5 Council từ chối đề xuất không đạt
Thử inject một đề xuất thuốc và một bằng chứng ID giả. Đây là kiểm tra phần mềm có chủ đích; không phải đo tỷ lệ sự cố lâm sàng.

In [6]:
bad = [{"kind": "medication", "status": "completed"},
       {"kind": "knowledge", "status": "completed", "evidence": [{"id": "FAKE"}]}]
veto = review(bad, {d["id"] for d in system.kb.docs})
print(veto)
assert veto["status"] == "rejected"

{'status': 'rejected', 'reasons': ['Không cho phép đề xuất thuốc hoặc liều điều trị.', 'Bằng chứng chứa mã nguồn không hợp lệ.']}


## 6 Chạy bộ đánh giá và xem kết quả
24 câu truy xuất, 20 kịch bản hệ thống và 6 lỗi inject đều do tác giả thiết kế. Phải phân biệt tỷ lệ đạt hợp đồng với chất lượng y khoa hoặc độ đúng hội thoại mở.

In [7]:
summary = evaluate_all()
print(summary)
display(pd.read_csv(ARTIFACTS / "system_evaluation.csv"))
display(pd.read_csv(ARTIFACTS / "council_ablation.csv"))
assert summary["system_passed"] == summary["system_cases"]

{'system_cases': 20, 'system_passed': 20, 'latency_median_ms': 0.06780005060136318, 'retrieval': {False: {'hit_at_1': 1.0, 'hit_at_3': 1.0, 'reciprocal_rank': 1.0}, True: {'hit_at_1': 1.0, 'hit_at_3': 1.0, 'reciprocal_rank': 1.0}}, 'faults_rejected': 6, 'limitations': 'Authored 24-query retrieval and 20-case contract suites. Fault injection is constructed, not a clinical safety rate.'}


,case,expected,actual,passed,messages,elapsed_ms
0,risk_complete,completed,completed,True,8,62.1554
1,risk_mlp,completed,completed,True,8,4.1628
2,risk_missing,needs_input,needs_input,True,8,0.0705
3,risk_invalid,needs_input,needs_input,True,8,0.0520
4,risk_nan,needs_input,needs_input,True,8,0.0408
5,knowledge_a1c,completed,completed,True,8,2.3952
6,knowledge_food,completed,completed,True,8,2.1738
7,knowledge_activity,completed,completed,True,8,2.2371
8,knowledge_unknown,insufficient_evidence,insufficient_evidence,True,8,2.5753
9,knowledge_blank,insufficient_evidence,insufficient_evidence,True,8,0.0478


,fault,with_council,without_council
0,1,rejected,unreviewed
1,2,rejected,unreviewed
2,3,rejected,unreviewed
3,4,rejected,unreviewed
4,5,rejected,unreviewed
5,6,rejected,unreviewed


## 7 Chạy giao diện local
Mở terminal tại PROJECT_01 và chạy lệnh dưới. Lệnh được in ra để tránh notebook giữ kernel vì server. Trong giao diện thử bốn tác vụ, đổi sang MLP, tải trace JSON và xem benchmark.

In [8]:
print("python -m streamlit run app.py --server.address 127.0.0.1 --server.port 8521")
print("Mở http://127.0.0.1:8521 | Dừng server bằng Ctrl+C")

python -m streamlit run app.py --server.address 127.0.0.1 --server.port 8521
Mở http://127.0.0.1:8521 | Dừng server bằng Ctrl+C
